# 第07回 AfDesignで配列設計

**目標は、指定した骨格を再現する新しいアミノ酸配列を作ること。** 目標はPDB **1TENの鎖A**（座標がある89残基）。骨格と学習済みAlphaFold2の重みを固定し、入力する配列側の得点を更新する。

計算の順序は **準備 → 設計 → 最適化の推移 → 目標との重ね合わせ → 保存**。各セルを上から実行する。途中の設問・提出はない。

- 「ランタイム → ランタイムのタイプを変更」で **T4 GPU** を選ぶ。
- 短い演習設定は `soft 100 / temp 30 / hard 10`、`seed 42`、AF2 model 1、recycle 0。演習の目安は20分。初回の重み取得とGPU待ち時間は別に変動する。
- 目標構造と一致するかはRMSD、モデルの信頼度はpLDDTで読む。実験での発現・安定性・機能は別の検証が必要。
- Colab上で表示するため、ファイルをダウンロードして別のソフトへ移す必要はない。最後に結果一式を保存できる。

最後の発展欄では、PDB IDと鎖を指定して、1TEN以外の骨格でも配列設計を試せます。


## 1. GPUと計算環境を準備する

GPUの確認と、演習専用Python環境の作成を行う。


In [ ]:
from pathlib import Path
import subprocess,sys,os,time,json,urllib.request
start=time.perf_counter()
work=Path('/content/bioinfo07');work.mkdir(exist_ok=True);os.chdir(work)
r=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
assert r.returncode==0,'ランタイムをT4 GPUへ変更してください。'
print(r.stdout)
subprocess.run([sys.executable,'-m','pip','-q','install','uv','py3Dmol==2.5.3'],check=True)
py=str(work/'.venv/bin/python')
if not Path(py).exists():subprocess.run(['uv','venv','--python','3.11',str(work/'.venv')],check=True)


## 2. AF2の重みと目標構造を取得する

学習済みモデルとPDB 1TENを取得する。初回はダウンロードに時間がかかる。


In [ ]:
# 初回はGPU用ライブラリと学習済みパラメータを取得します。
subprocess.run(['uv','pip','install','--python',py,
 'numpy==1.26.4','scipy==1.12.0','jax[cuda12]==0.4.35','nvidia-cuda-nvcc-cu12==12.6.85','dm-haiku==0.0.13','optax==0.2.4','chex==0.1.88',
 'git+https://github.com/sokrypton/ColabDesign.git@e31a56fe1d9b4de25c8697f3a28b75892941cc72'],check=True)
params=work/'params';params.mkdir(exist_ok=True)
if not (params/'params_model_1_ptm.npz').exists():
    archive=work/'alphafold_params_2022-12-06.tar'
    subprocess.run(['wget','-q','-c','https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar','-O',str(archive)],check=True)
    import tarfile
    with tarfile.open(archive) as tar:
        member=next(m for m in tar.getmembers() if Path(m.name).name=='params_model_1_ptm.npz')
        with tar.extractfile(member) as src, open(params/'params_model_1_ptm.npz','wb') as dst:
            import shutil;shutil.copyfileobj(src,dst)
urllib.request.urlretrieve('https://files.rcsb.org/download/1TEN.pdb',work/'1TEN.pdb')
print('準備秒',round(time.perf_counter()-start))


## 3. 目標骨格を固定して配列を最適化する

変えるのは、各位置の20種類のアミノ酸の得点（ロジット）。学習済みAF2の重みと目標骨格は変えない。

1. **logits → soft**：実数の得点からsoftmaxの重みを使う入力へ移す（100回）。
2. **温度を下げる**：候補の重みを鋭くする（30回）。
3. **hard**：各位置1種類の配列をAF2へ渡し、連続表現を使う勾配で更新する（10回）。

この固定骨格設計の損失 `dgram_cce` は、目標の残基間距離の区間にAF2が与える確率の交差エントロピー。小さいほど目標の距離と整合する。RMSDそのものを損失としているわけではない。

実行は独立したPython環境で行う。最初はJAXのコンパイルに時間がかかる。ログが出るまで同じセルを再実行せず待つ。


In [ ]:
design_script = "import os\nos.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='false'\nos.environ['MPLBACKEND']='Agg'\nimport json,time\nfrom pathlib import Path\nimport numpy as np\nimport jax\nimport matplotlib;matplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nfrom colabdesign import mk_afdesign_model\nassert any(d.platform=='gpu' for d in jax.devices()), 'GPUが使えません'\nprint('devices',jax.devices(),flush=True)\ncfg=json.loads(Path('input.json').read_text())\nt0=time.perf_counter()\nmodel=mk_afdesign_model(protocol='fixbb',data_dir=cfg['params'],model_names=['model_1_ptm'],num_recycles=0)\nmodel.prep_inputs(pdb_filename=cfg['target_file'],chain=cfg['chain'])\nmodel.restart(seed=cfg['seed'])\nmodel.design_3stage(soft_iters=100,temp_iters=30,hard_iters=10,num_models=1)\nseq=model.get_seqs()[0]\nmodel.save_pdb('design.pdb')\nmodel.plot_traj();plt.savefig('trajectory.png',dpi=160,bbox_inches='tight');plt.close('all')\nreport={'sequence':seq,'length':len(seq),'seconds':time.perf_counter()-t0,'conditions':{**cfg,'soft':100,'temp':30,'hard':10,'model':'model_1_ptm','num_recycles':0},'last_step_metrics':{k:float(np.asarray(v).reshape(-1)[0]) for k,v in model.aux['log'].items() if np.asarray(v).size==1}}\nPath('result.json').write_text(json.dumps(report,indent=2))\nPath('design.fasta').write_text('>bioinfo07_fixbb_'+cfg['pdb']+'_'+cfg['chain']+'\\n'+seq+'\\n')\nprint(json.dumps(report),flush=True)\n"


def run_design(run_dir, pdb_id, chain, target_file, seed=42):
    run_dir=Path(run_dir)
    cfg={'pdb':pdb_id,'chain':chain,'target_file':str(Path(target_file).resolve()),
         'params':str(params.resolve()),'seed':int(seed)}
    (run_dir/'input.json').write_text(json.dumps(cfg,indent=2))
    (run_dir/'run_design.py').write_text(design_script)
    process=subprocess.Popen([py,'run_design.py'],cwd=run_dir,
                             stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout: print(line,end='',flush=True)
    if process.wait()!=0:
        raise RuntimeError('設計が完了しませんでした。上のログを確認してください。')
    print('設計完了:',pdb_id,'鎖',chain)

run_design(work,'1TEN','A',work/'1TEN.pdb')


## 4. 最適化の推移と重ね合わせを見る

下のセルは、設計結果を表示し、保存した構造を目標1TENへ重ね合わせる。灰色＝目標、虹色＝設計配列の予測構造。虹色は配列位置を表す。

軌跡図の構造は青ほど高いpLDDT、PAE行列は青ほど小さい予測配置誤差。下の配列行列は横＝位置、縦＝アミノ酸の種類、濃さ＝重み。


In [ ]:
def show_result(run_dir, target_file, chain):
    run_dir=Path(run_dir)
    from IPython.display import display, Image
    report=json.loads((run_dir/'result.json').read_text())
    print(json.dumps(report,ensure_ascii=False,indent=2))
    display(Image(filename=str(run_dir/'trajectory.png')))
    import py3Dmol
    v=py3Dmol.view(width=850,height=500)
    # Cαを残基順に対応づけ、剛体重ね合わせする。
    import numpy as np
    ref_lines=[l for l in Path(target_file).read_text().splitlines() if l.startswith('ATOM') and l[21]==chain]
    pred_lines=(run_dir/'design.pdb').read_text().splitlines()
    def ca(lines):
        return np.array([[float(l[30:38]),float(l[38:46]),float(l[46:54])] for l in lines if l.startswith('ATOM') and l[12:16].strip()=='CA' and l[16] in [' ','A']])
    a=ca(pred_lines);b=ca(ref_lines)
    assert a.shape==b.shape, '残基対応を確認してください'
    u,_,vt=np.linalg.svd((a-a.mean(0)).T@(b-b.mean(0)))
    d=np.eye(3);d[-1,-1]=np.sign(np.linalg.det(u@vt));rot=u@d@vt
    aligned=(a-a.mean(0))@rot+b.mean(0)
    saved_rmsd=float(np.sqrt(np.mean(np.sum((aligned-b)**2,axis=1))))
    print('保存した構造のCα RMSD [Å]:',saved_rmsd)
    report=json.loads((run_dir/'result.json').read_text());report['saved_structure_ca_rmsd_A']=saved_rmsd
    report['metrics_note']='last_step_metricsは最終反復、保存構造と配列は保存対象の最良候補。RMSDは保存PDBから計算。'
    (run_dir/'result.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
    out=[]
    for l in pred_lines:
        if l.startswith('ATOM'):
            q=(np.array([float(l[30:38]),float(l[38:46]),float(l[46:54])])-a.mean(0))@rot+b.mean(0)
            l=l[:30]+''.join(f'{v:8.3f}' for v in q)+l[54:]
        out.append(l)
    (run_dir/'aligned.pdb').write_text(chr(10).join(out))
    v.addModel(chr(10).join(ref_lines),'pdb');v.setStyle({'model':0},{'cartoon':{'color':'gray','opacity':0.45}})
    v.addModel(chr(10).join(out),'pdb');v.setStyle({'model':1},{'cartoon':{'color':'spectrum'}})
    v.zoomTo();v.show()
    print('灰色：目標構造、虹色：設計配列の予測構造')


show_result(work,work/'1TEN.pdb','A')


## 結果の読み方

| 表示 | 意味 | 読み方 |
|---|---|---|
| `loss` / `dgram_cce` | 目標の距離区間とAF2の予測確率のずれ | 小さいほど目標に整合 |
| 保存PDBのCα RMSD | 目標と89残基のCαを重ね合わせた後の座標のずれ | Å単位。小さいほど形が一致 |
| `plddt` | AF2が推定した局所構造の信頼度 | ログは0–1。0.85なら通常の85 |

立体表示では**灰色が目標の1TEN、虹色が保存した設計配列の予測構造**。ドラッグで回転し、主鎖とループのどこが一致し、どこがずれるかを見る。虹色は配列位置の色であり、pLDDTの色ではない。

`last_step_metrics` は**最終反復のログ**。`design.pdb` は最適化中に保存された候補なので、両者は同じとは限らない。構造を評価するときは、このセルがPDBから再計算した `saved_structure_ca_rmsd_A` を読む。

反復数を変える場合は同じseedで比較し、損失だけでなく、離散配列にした後の形も確認する。高いpLDDTだけで実験成功とは判断しない。


## 5. 配列・構造・計算条件を保存する

ZIPに配列、予測PDB、重ね合わせたPDB、条件・指標、最適化の図をまとめる。


In [ ]:
import zipfile
from google.colab import files

def save_results(run_dir, target_file, zip_name):
    run_dir=Path(run_dir)
    archive=run_dir/zip_name
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        z.write(target_file,arcname='target.pdb')
        for name in ['design.pdb','design.fasta','result.json','trajectory.png',
                     'run_design.py','aligned.pdb','input.json']:
            z.write(run_dir/name,arcname=name)
    files.download(str(archive))

save_results(work,work/'1TEN.pdb','bioinfo07_results.zip')


## 6. 自分で目標構造を選んで設計する（発展）

今度は、**PDB IDと鎖を変更して、その骨格に合う配列を設計する**。例として `1UBQ` の鎖 `A`（ユビキチン、76残基）を入力してある。自分で選んだ構造へ変更してよい。

1. 下の **pdb_id** と **chain_id** を入力し、「目標構造を確認する」を実行する。
2. 表示されたPDB ID・鎖・残基数と立体構造を確認する。
3. 「選んだ構造で配列を設計する」を実行する。1TENと同じ反復数・seedで計算する。
4. 結果の重ね合わせとRMSDを確認し、必要ならZIPを保存する。

授業では計算量を抑えるため、**30〜200残基のタンパク質1鎖**を対象とする。PDB形式で取得できる構造を使い、複数モデルがある場合は最初のモデルだけを使う。水・低分子・ほかの鎖は設計条件に含めない。主鎖原子の欠けや非標準残基がある場合は、別の構造を選ぶ。PDBに座標がない残基は補わない。

1TENの結果は残り、別の目標の結果は実行ごとのフォルダに保存される。環境構築とAF2の重み取得をやり直す必要はない。**フォームを変更したら、必ず確認セルから再実行する。**

In [ ]:
#@title 目標構造を確認する
pdb_id = "1UBQ" #@param {type:"string"}
chain_id = "A" #@param {type:"string"}

import re, math
from collections import OrderedDict
from urllib.error import HTTPError, URLError
from datetime import datetime

# 取得に失敗した場合、前の目標や結果を新しいものとして使わない。
custom_target=None
custom_result=None

def prepare_target(pdb_id, chain_id):
    pdb_id=pdb_id.strip().upper()
    chain_id=chain_id.strip()
    if not re.fullmatch(r'[0-9][A-Z0-9]{3}',pdb_id):
        raise ValueError('PDB IDは1UBQのような4文字で入力してください。')
    if not re.fullmatch(r'[A-Za-z0-9]',chain_id):
        raise ValueError('鎖はAなど1文字で指定してください。大文字と小文字を区別します。')
    try:
        with urllib.request.urlopen(f'https://files.rcsb.org/download/{pdb_id}.pdb',timeout=30) as response:
            text=response.read().decode('utf-8')
    except (HTTPError,URLError,TimeoutError) as exc:
        raise ValueError('PDB形式の構造を取得できません。PDB IDと通信状態を確認してください。') from exc
    # 最初のモデルに含まれるATOMレコードだけを選ぶ。
    lines=[]
    for line in text.splitlines():
        if line.startswith('ENDMDL'): break
        if line.startswith('ATOM  '): lines.append(line)
    chains=sorted({line[21] for line in lines})
    if chain_id not in chains:
        raise ValueError(f'鎖{chain_id}が見つかりません。選べる鎖: {", ".join(chains)}')
    residues=OrderedDict()
    for line in lines:
        if line[21]==chain_id:
            residues.setdefault(line[22:27],[]).append(line)
    standard=set('ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL'.split())
    selected=[]
    skipped=[]
    length=0
    for resid,atoms in residues.items():
        # 代替配置がある場合は占有率の合計が最大の配置を採用する。
        labels=sorted({line[16] for line in atoms if line[16]!=' '})
        occupancy=lambda line: float(line[54:60].strip() or 0)
        alt=max(labels,key=lambda label:sum(occupancy(l) for l in atoms if l[16]==label)) if labels else ' '
        chosen={}
        for line in atoms:
            if line[16] in (' ',alt):
                name=line[12:16].strip()
                if name not in chosen or line[16]==' ': chosen[name]=line
        if 'CA' not in chosen:
            skipped.append(resid.strip())
            continue
        length+=1
        if not {'N','CA','C','O'}<=set(chosen):
            raise ValueError(f'残基{resid.strip()}の主鎖原子が欠けています。別の構造を選んでください。')
        if any(l[17:20].strip() not in standard for l in chosen.values()):
            raise ValueError(f'残基{resid.strip()}が標準アミノ酸ではありません。別の構造を選んでください。')
        for line in chosen.values():
            xyz=[float(line[a:b]) for a,b in [(30,38),(38,46),(46,54)]]
            if not all(math.isfinite(v) for v in xyz): raise ValueError('座標に不正な値があります。')
            selected.append(line[:16]+' '+line[17:])
    if skipped: print("Cαの座標がないため除外する残基:", ", ".join(skipped))
    if not 30<=length<=200:
        raise ValueError(f'鎖{chain_id}は{length}残基です。授業では30〜200残基の鎖を選んでください。')
    # 修飾アミノ酸などを気付かず落としたまま設計しない。
    if any(l.startswith('HETATM') and l[21]==chain_id and l[12:16].strip()=='CA'
           and l[17:20].strip()!='CA' for l in text.split('ENDMDL')[0].splitlines()):
        raise ValueError('鎖内に修飾残基の可能性があります。標準アミノ酸のみの構造を選んでください。')
    target_dir=work/'custom_targets'/f'{pdb_id}_{chain_id}'
    target_dir.mkdir(parents=True,exist_ok=True)
    target_file=target_dir/'target.pdb'
    target_file.write_text('\n'.join(selected)+'\nTER\nEND\n')
    return {'pdb':pdb_id,'chain':chain_id,'length':length,'file':target_file}

custom_target=prepare_target(pdb_id,chain_id)
print(f"目標：{custom_target['pdb']} 鎖{custom_target['chain']}、{custom_target['length']}残基")
import py3Dmol
preview=py3Dmol.view(width=850,height=450)
preview.addModel(custom_target['file'].read_text(),'pdb')
preview.setStyle({'cartoon':{'color':'spectrum'}})
preview.zoomTo();preview.show()
print('この構造を目標にする場合は、次の設計セルを実行してください。')

In [ ]:
#@title 選んだ構造で配列を設計する
custom_result=None
if custom_target is None:
    raise ValueError('先に「目標構造を確認する」を実行してください。')
import shutil
run_dir=work/'custom_results'/f"{custom_target['pdb']}_{custom_target['chain']}_{datetime.now():%Y%m%d_%H%M%S_%f}"
run_dir.mkdir(parents=True,exist_ok=False)
target_copy=run_dir/'target.pdb'
shutil.copy2(custom_target['file'],target_copy)
print(f"設計中：{custom_target['pdb']} 鎖{custom_target['chain']}、{custom_target['length']}残基")
run_design(run_dir,custom_target['pdb'],custom_target['chain'],target_copy,seed=42)
custom_result={**custom_target,'dir':run_dir,'file':target_copy}
show_result(run_dir,target_copy,custom_result['chain'])

In [ ]:
#@title 別の目標で設計した結果を保存する
if custom_result is None:
    raise ValueError('先に、選んだ構造での設計を完了してください。')
save_results(custom_result['dir'],custom_result['file'],
             f"bioinfo07_{custom_result['pdb']}_{custom_result['chain']}_results.zip")

## 再利用元
公式[ColabDesign AfDesign](https://github.com/sokrypton/ColabDesign/blob/e31a56fe1d9b4de25c8697f3a28b75892941cc72/af/design.ipynb)。授業ではfixbbだけを使用し、依存環境と反復数を固定しています。

Frank et al. (2024), Scalable protein design using optimization in a relaxed sequence space. https://doi.org/10.1126/science.adq1741

ColabDesignのコードと学習済みパラメータは元のライセンスに従います。
